## Buy n hold 1/N

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
years = range(2014, 2025)
k = 5
metrics = ["hrm", "pozzi"]
# Todos os pares de decis
decile_pairs = [(1,10), (2,9), (3,8), (4,7), (5,6)] 

c_tc = 0.001 # Custo de transação (10 basis points) - Altere conforme sua metodologia

# 1. Carregamento do Universo Total
df_ret_full = pd.read_parquet("../../data/01_raw/returns.parquet")
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2015, 1, 1)]
if 'HYFT' in df_ret_full.columns:
    df_ret_full = df_ret_full.drop(columns=['HYFT'])

# =========================================================================
# BENCHMARK
# =========================================================================
benchmark_anual = []
w_last_benchmark = pd.Series(dtype=float) # Salva os pesos do fim do ano anterior

for year in df_ret_full.index.year.unique():
    df_ano = df_ret_full[df_ret_full.index.year == year]
    
    valid_cols = df_ano.dropna(axis=1, how='all').columns
    
    df_ano_filled = df_ano[valid_cols].fillna(0)
    
    cum_ret_ativos = (1 + df_ano_filled).cumprod()
    
    port_cum_ret = cum_ret_ativos.mean(axis=1)
    
    ret = port_cum_ret.pct_change()
    ret.iloc[0] = port_cum_ret.iloc[0] - 1
    
    # --- CÁLCULO DE TURNOVER E CUSTO DE TRANSAÇÃO (BENCHMARK) ---
    w_new = pd.Series(1.0 / len(valid_cols), index=valid_cols)
    turnover = w_new.sub(w_last_benchmark, fill_value=0).abs().sum()
    tc = c_tc * turnover
    
    # Subtrai o custo de transação do retorno do primeiro dia
    ret.iloc[0] = ret.iloc[0] - tc
    
    benchmark_anual.append(ret)
    
    # Atualiza os pesos finais da carteira no último dia útil para o ano seguinte
    w_last_benchmark = cum_ret_ativos.iloc[-1] / cum_ret_ativos.iloc[-1].sum()

market_benchmark_global = pd.concat(benchmark_anual).sort_index()
cum_ret_global = (1 + market_benchmark_global).cumprod()

# =========================================================================
# ESTRATÉGIAS
# =========================================================================
results_strat = {m: {f"d{p}_d{c}": [] for p, c in decile_pairs} for m in metrics}
# Dicionário extra para rastrear os pesos antigos de cada par de decil
w_last_strat = {m: {f"d{p}_d{c}": pd.Series(dtype=float) for p, c in decile_pairs} for m in metrics}

for year in years:
    test_year = year + 1
    df_ret_oos = df_ret_full[df_ret_full.index.year == test_year]
    
    if df_ret_oos.empty:
        continue

    for metric in metrics:
        for c_idx, p_idx in decile_pairs:
            chave_par = f"d{c_idx}_d{p_idx}"
            
            c_label = f"decil_{c_idx}_{year}_{metric}"
            p_label = f"decil_{p_idx}_{year}_{metric}"
            
            cols_c = pd.read_parquet(f"../../data/06_portfolios/{c_label}.parquet").columns
            cols_p = pd.read_parquet(f"../../data/06_portfolios/{p_label}.parquet").columns

            cols_total = (set(cols_c).union(set(cols_p)))
            
            valid = list(cols_total.intersection(df_ret_oos.columns))
            
            if len(valid) == 0:
                continue

            df_ret_filled = df_ret_oos[valid].fillna(0)
            
            cum_ret_ativos = (1 + df_ret_filled).cumprod()
            port_cum_ret = cum_ret_ativos.mean(axis=1)
            
            ret = port_cum_ret.pct_change()
            ret.iloc[0] = port_cum_ret.iloc[0] - 1
            
            # --- CÁLCULO DE TURNOVER E CUSTO DE TRANSAÇÃO (ESTRATÉGIA) ---
            w_new = pd.Series(1.0 / len(valid), index=valid)
            turnover = w_new.sub(w_last_strat[metric][chave_par], fill_value=0).abs().sum()
            tc = c_tc * turnover
            
            # Subtrai o TC do primeiro dia
            ret.iloc[0] = ret.iloc[0] - tc
            
            # Transforma em log-retorno para consistência
            ret = np.log1p(ret)

            results_strat[metric][chave_par].append(ret)
            
            # Atualiza os pesos finais para usar no ano seguinte
            w_last_strat[metric][chave_par] = cum_ret_ativos.iloc[-1] / cum_ret_ativos.iloc[-1].sum()

# Novo dicionário para armazenar as séries completas concatenadas
retornos_consolidados = {}
for metric in metrics:
    retornos_consolidados[metric] = {}
    for c_idx, p_idx in decile_pairs:
        chave_par = f"d{c_idx}_d{p_idx}"
        
        if len(results_strat[metric][chave_par]) > 0:
            serie_completa = pd.concat(results_strat[metric][chave_par])
            retornos_consolidados[metric][chave_par] = serie_completa.sort_index()

# Visualiza as consolidações
print(retornos_consolidados["hrm"]["d1_d10"].head())

Date
2015-01-02   -0.001923
2015-01-05   -0.010727
2015-01-06   -0.010417
2015-01-07    0.005261
2015-01-08    0.007277
dtype: float64


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

def calculate_metrics(returns_simple, series_name):
    """
    Calculates financial metrics from a pandas Series of daily simple returns.
    """
    # Exige no mínimo 1 ano de dados
    if len(returns_simple) < 252:
        return pd.Series(dtype=float)
        
    cum_wealth = (1 + returns_simple).cumprod()
    
    # 1. CAGR
    years = len(returns_simple) / 252.0
    cagr = (cum_wealth.iloc[-1] ** (1 / years)) - 1
    
    # 2. Annual Volatility
    vol_annual = returns_simple.std() * np.sqrt(252)
    
    # 3. Sharpe Ratio (assuming Risk-Free = 0)
    sharpe = (returns_simple.mean() * 252) / vol_annual if vol_annual != 0 else 0
    
    # 4. Sortino Ratio
    neg_returns = returns_simple[returns_simple < 0]
    downside_std = neg_returns.std() * np.sqrt(252)
    sortino = (returns_simple.mean() * 252) / downside_std if downside_std != 0 else 0
    
    # 5. Drawdowns
    highwater_mark = cum_wealth.cummax()
    drawdown = (cum_wealth / highwater_mark) - 1
    max_dd = drawdown.min()
    
    underwater = drawdown[drawdown < 0]
    avg_dd = underwater.mean() if len(underwater) > 0 else 0
        
    # 6. VaR 99% and CVaR 99%
    var_99 = np.percentile(returns_simple, 1)
    cvar_99 = returns_simple[returns_simple <= var_99].mean()
    
    metrics = {
        "CAGR": cagr,
        "Annual Volatility": vol_annual,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_dd,
        "Average Drawdown": avg_dd,
        "VaR 99%": var_99,
        "CVaR 99%": cvar_99
    }
    
    return pd.Series(metrics, name=series_name)

# ==========================================
# Generate Tables for HCM and Pozzi
# ==========================================

# 1. Benchmark metrics
bench_simple = market_benchmark_global
metrics_bench = calculate_metrics(bench_simple, "Global Benchmark")

# 2. HCM Metrics Table
list_hcm = [metrics_bench]
for c_idx, p_idx in decile_pairs:
    chave_par = f"d{c_idx}_d{p_idx}"
    if chave_par in retornos_consolidados["hrm"]:
        # Converte de volta de log-retorno para retorno simples (expm1 é o inverso de log1p)
        strat_simple = np.expm1(retornos_consolidados["hrm"][chave_par])
        list_hcm.append(calculate_metrics(strat_simple, f"Deciles {c_idx}-{p_idx}"))

df_hcm = pd.DataFrame(list_hcm)

# 3. Pozzi Metrics Table
list_pozzi = [metrics_bench]
for c_idx, p_idx in decile_pairs:
    chave_par = f"d{c_idx}_d{p_idx}"
    if chave_par in retornos_consolidados["pozzi"]:
        strat_simple = np.expm1(retornos_consolidados["pozzi"][chave_par])
        list_pozzi.append(calculate_metrics(strat_simple, f"Deciles {c_idx}-{p_idx}"))

df_pozzi = pd.DataFrame(list_pozzi)

# ==========================================
# Formatting and Display (Versão sem Jinja2)
# ==========================================

def format_table_simple(df):
    # Formata as colunas diretamente usando pd.Series.apply
    df_fmt = pd.DataFrame()
    df_fmt["CAGR"] = df["CAGR"].apply(lambda x: f"{x:.2%}")
    df_fmt["Annual Volatility"] = df["Annual Volatility"].apply(lambda x: f"{x:.2%}")
    df_fmt["Sharpe Ratio"] = df["Sharpe Ratio"].apply(lambda x: f"{x:.2f}")
    df_fmt["Sortino Ratio"] = df["Sortino Ratio"].apply(lambda x: f"{x:.2f}")
    df_fmt["Max Drawdown"] = df["Max Drawdown"].apply(lambda x: f"{x:.2%}")
    df_fmt["Average Drawdown"] = df["Average Drawdown"].apply(lambda x: f"{x:.2%}")
    df_fmt["VaR 99%"] = df["VaR 99%"].apply(lambda x: f"{x:.2%}")
    df_fmt["CVaR 99%"] = df["CVaR 99%"].apply(lambda x: f"{x:.2%}")
    
    # Mantém o index original (nome da estratégia)
    df_fmt.index = df.index
    return df_fmt

print("--- HCM Strategy Metrics ---")
df_hcm.index = df_hcm.index.rename("Strategy")
display(format_table_simple(df_hcm))

print("\n--- Pozzi Strategy Metrics ---")
df_pozzi.index = df_pozzi.index.rename("Strategy")
display(format_table_simple(df_pozzi))

--- HCM Strategy Metrics ---


,CAGR,Annual Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Average Drawdown,VaR 99%,CVaR 99%
Strategy,,,,,,,,
Global Benchmark,11.04%,15.61%,0.75,0.93,-34.34%,-6.56%,-2.55%,-3.98%
Deciles 1-10,11.96%,13.14%,0.93,1.26,-27.21%,-6.30%,-2.13%,-3.02%
Deciles 2-9,11.61%,16.38%,0.75,0.97,-34.67%,-7.52%,-2.69%,-4.07%
Deciles 3-8,8.97%,17.18%,0.59,0.72,-39.34%,-7.01%,-2.75%,-4.45%
Deciles 4-7,10.69%,17.23%,0.68,0.84,-38.74%,-6.44%,-2.85%,-4.49%
Deciles 5-6,10.31%,16.92%,0.67,0.82,-38.10%,-5.88%,-2.81%,-4.44%



--- Pozzi Strategy Metrics ---


,CAGR,Annual Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Average Drawdown,VaR 99%,CVaR 99%
Strategy,,,,,,,,
Global Benchmark,11.04%,15.61%,0.75,0.93,-34.34%,-6.56%,-2.55%,-3.98%
Deciles 1-10,12.37%,15.27%,0.84,1.09,-34.06%,-5.81%,-2.48%,-3.80%
Deciles 2-9,10.41%,15.00%,0.74,0.93,-33.51%,-6.58%,-2.52%,-3.87%
Deciles 3-8,9.34%,15.89%,0.64,0.82,-34.48%,-6.88%,-2.57%,-3.95%
Deciles 4-7,11.36%,18.07%,0.69,0.86,-37.49%,-7.30%,-2.97%,-4.60%
Deciles 5-6,9.96%,17.29%,0.64,0.79,-39.23%,-7.17%,-2.76%,-4.41%


## DECIS

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
years = range(2014, 2025)
k = 5
metrics = ["hrm", "pozzi"]
# Lista com todos os decis individuais
deciles = list(range(1, 11))

c_tc = 0.001 # Custo de transação (10 basis points) - Altere conforme sua metodologia

# 1. Carregamento do Universo Total
df_ret_full = pd.read_parquet("../../data/01_raw/returns.parquet")
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2015, 1, 1)]
if 'HYFT' in df_ret_full.columns:
    df_ret_full = df_ret_full.drop(columns=['HYFT'])

# =========================================================================
# BENCHMARK
# =========================================================================
benchmark_anual = []
w_last_benchmark = pd.Series(dtype=float) # Salva os pesos do fim do ano anterior

for year in df_ret_full.index.year.unique():
    df_ano = df_ret_full[df_ret_full.index.year == year]
    
    valid_cols = df_ano.dropna(axis=1, how='all').columns
    df_ano_filled = df_ano[valid_cols].fillna(0)
    cum_ret_ativos = (1 + df_ano_filled).cumprod()
    port_cum_ret = cum_ret_ativos.mean(axis=1)
    
    ret = port_cum_ret.pct_change()
    ret.iloc[0] = port_cum_ret.iloc[0] - 1
    
    # --- CÁLCULO DE TURNOVER E CUSTO DE TRANSAÇÃO (BENCHMARK) ---
    w_new = pd.Series(1.0 / len(valid_cols), index=valid_cols)
    turnover = w_new.sub(w_last_benchmark, fill_value=0).abs().sum()
    tc = c_tc * turnover
    
    # Subtrai o custo de transação do retorno do primeiro dia
    ret.iloc[0] = ret.iloc[0] - tc
    benchmark_anual.append(ret)
    
    # Atualiza os pesos finais da carteira no último dia útil para o ano seguinte
    w_last_benchmark = cum_ret_ativos.iloc[-1] / cum_ret_ativos.iloc[-1].sum()

market_benchmark_global = pd.concat(benchmark_anual).sort_index()
cum_ret_global = (1 + market_benchmark_global).cumprod()

# =========================================================================
# ESTRATÉGIAS
# =========================================================================
results_strat = {m: {f"decil_{d}": [] for d in deciles} for m in metrics}
# Dicionário extra para rastrear os pesos antigos de cada decil
w_last_strat = {m: {f"decil_{d}": pd.Series(dtype=float) for d in deciles} for m in metrics}

for year in years:
    test_year = year + 1
    df_ret_oos = df_ret_full[df_ret_full.index.year == test_year]
    
    if df_ret_oos.empty:
        continue

    for metric in metrics:
        for decil in deciles:
            chave_decil = f"decil_{decil}"
            
            # Trata nomenclatura hrm/hcm
            metric_file_name = "hcm" if metric == "hrm" else metric 
            d_label = f"decil_{decil}_{year}_{metric_file_name}"
            
            try:
                cols_decil = pd.read_parquet(f"../../data/06_portfolios/{d_label}.parquet").columns
            except FileNotFoundError:
                # Fallback caso o arquivo ainda use 'hrm'
                d_label_fallback = f"decil_{decil}_{year}_{metric}"
                cols_decil = pd.read_parquet(f"../../data/06_portfolios/{d_label_fallback}.parquet").columns
            
            valid = list(set(cols_decil).intersection(df_ret_oos.columns))
            
            if len(valid) == 0:
                continue

            df_ret_filled = df_ret_oos[valid].fillna(0)
            
            cum_ret_ativos = (1 + df_ret_filled).cumprod()
            port_cum_ret = cum_ret_ativos.mean(axis=1)
            
            ret = port_cum_ret.pct_change()
            ret.iloc[0] = port_cum_ret.iloc[0] - 1
            
            # --- CÁLCULO DE TURNOVER E CUSTO DE TRANSAÇÃO (ESTRATÉGIA) ---
            w_new = pd.Series(1.0 / len(valid), index=valid)
            turnover = w_new.sub(w_last_strat[metric][chave_decil], fill_value=0).abs().sum()
            tc = c_tc * turnover
            
            # Subtrai o TC do primeiro dia
            ret.iloc[0] = ret.iloc[0] - tc
            
            # Transforma em log-retorno para consistência
            ret = np.log1p(ret)

            results_strat[metric][chave_decil].append(ret)
            
            # Atualiza os pesos finais para usar no ano seguinte
            w_last_strat[metric][chave_decil] = cum_ret_ativos.iloc[-1] / cum_ret_ativos.iloc[-1].sum()

# Novo dicionário para armazenar as séries completas concatenadas
retornos_consolidados = {}
for metric in metrics:
    retornos_consolidados[metric] = {}
    for decil in deciles:
        chave_decil = f"decil_{decil}"
        
        if len(results_strat[metric][chave_decil]) > 0:
            serie_completa = pd.concat(results_strat[metric][chave_decil])
            retornos_consolidados[metric][chave_decil] = serie_completa.sort_index()

# =========================================================================
# TABELAS DE MÉTRICAS (DECIS INDIVIDUAIS)
# =========================================================================
def calculate_metrics(returns_simple, series_name):
    """
    Calculates financial metrics from a pandas Series of daily simple returns.
    """
    if len(returns_simple) < 252:
        return pd.Series(dtype=float)
        
    cum_wealth = (1 + returns_simple).cumprod()
    
    # 1. CAGR
    years_count = len(returns_simple) / 252.0
    cagr = (cum_wealth.iloc[-1] ** (1 / years_count)) - 1
    
    # 2. Annual Volatility
    vol_annual = returns_simple.std() * np.sqrt(252)
    
    # 3. Sharpe Ratio (assuming Risk-Free = 0)
    sharpe = (returns_simple.mean() * 252) / vol_annual if vol_annual != 0 else 0
    
    # 4. Sortino Ratio
    neg_returns = returns_simple[returns_simple < 0]
    downside_std = neg_returns.std() * np.sqrt(252)
    sortino = (returns_simple.mean() * 252) / downside_std if downside_std != 0 else 0
    
    # 5. Drawdowns
    highwater_mark = cum_wealth.cummax()
    drawdown = (cum_wealth / highwater_mark) - 1
    max_dd = drawdown.min()
    
    underwater = drawdown[drawdown < 0]
    avg_dd = underwater.mean() if len(underwater) > 0 else 0
        
    # 6. VaR 99% and CVaR 99%
    var_99 = np.percentile(returns_simple, 1)
    cvar_99 = returns_simple[returns_simple <= var_99].mean()
    
    metrics_dict = {
        "CAGR": cagr,
        "Annual Volatility": vol_annual,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_dd,
        "Average Drawdown": avg_dd,
        "VaR 99%": var_99,
        "CVaR 99%": cvar_99
    }
    
    return pd.Series(metrics_dict, name=series_name)

# 1. Benchmark metrics
bench_simple = market_benchmark_global
metrics_bench = calculate_metrics(bench_simple, "Global Benchmark")

# 2. HCM Metrics Table
list_hcm = [metrics_bench]
for decil in deciles:
    chave_decil = f"decil_{decil}"
    if chave_decil in retornos_consolidados["hrm"]:
        # Converte de volta de log-retorno para retorno simples (expm1 é o inverso de log1p)
        strat_simple = np.expm1(retornos_consolidados["hrm"][chave_decil])
        list_hcm.append(calculate_metrics(strat_simple, f"Decil {decil}"))

df_hcm = pd.DataFrame(list_hcm)

# 3. Pozzi Metrics Table
list_pozzi = [metrics_bench]
for decil in deciles:
    chave_decil = f"decil_{decil}"
    if chave_decil in retornos_consolidados["pozzi"]:
        strat_simple = np.expm1(retornos_consolidados["pozzi"][chave_decil])
        list_pozzi.append(calculate_metrics(strat_simple, f"Decil {decil}"))

df_pozzi = pd.DataFrame(list_pozzi)

def format_table_simple(df):
    df_fmt = pd.DataFrame()
    df_fmt["CAGR"] = df["CAGR"].apply(lambda x: f"{x:.2%}")
    df_fmt["Annual Volatility"] = df["Annual Volatility"].apply(lambda x: f"{x:.2%}")
    df_fmt["Sharpe Ratio"] = df["Sharpe Ratio"].apply(lambda x: f"{x:.2f}")
    df_fmt["Sortino Ratio"] = df["Sortino Ratio"].apply(lambda x: f"{x:.2f}")
    df_fmt["Max Drawdown"] = df["Max Drawdown"].apply(lambda x: f"{x:.2%}")
    df_fmt["Average Drawdown"] = df["Average Drawdown"].apply(lambda x: f"{x:.2%}")
    df_fmt["VaR 99%"] = df["VaR 99%"].apply(lambda x: f"{x:.2%}")
    df_fmt["CVaR 99%"] = df["CVaR 99%"].apply(lambda x: f"{x:.2%}")
    
    df_fmt.index = df.index
    return df_fmt

print("--- HCM Strategy Metrics ---")
df_hcm.index = df_hcm.index.rename("Strategy")
display(format_table_simple(df_hcm))

print("\n--- Pozzi Strategy Metrics ---")
df_pozzi.index = df_pozzi.index.rename("Strategy")
display(format_table_simple(df_pozzi))

--- HCM Strategy Metrics ---


,CAGR,Annual Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Average Drawdown,VaR 99%,CVaR 99%
Strategy,,,,,,,,
Global Benchmark,11.04%,15.61%,0.75,0.93,-34.34%,-6.56%,-2.55%,-3.98%
Decil 1,8.40%,18.26%,0.53,0.75,-34.36%,-7.10%,-3.14%,-4.01%
Decil 2,11.57%,19.65%,0.66,0.85,-40.10%,-6.04%,-3.20%,-4.92%
Decil 3,9.85%,18.67%,0.60,0.75,-40.92%,-6.21%,-3.01%,-4.76%
Decil 4,9.39%,18.01%,0.59,0.74,-40.15%,-6.55%,-2.89%,-4.67%
Decil 5,9.47%,16.82%,0.62,0.76,-39.10%,-5.90%,-2.79%,-4.46%
Decil 6,11.04%,17.44%,0.69,0.86,-37.36%,-6.12%,-2.91%,-4.50%
Decil 7,11.80%,17.22%,0.73,0.91,-37.34%,-7.20%,-2.93%,-4.42%
Decil 8,8.00%,16.57%,0.55,0.67,-37.80%,-9.18%,-2.65%,-4.33%



--- Pozzi Strategy Metrics ---


,CAGR,Annual Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Average Drawdown,VaR 99%,CVaR 99%
Strategy,,,,,,,,
Global Benchmark,11.04%,15.61%,0.75,0.93,-34.34%,-6.56%,-2.55%,-3.98%
Decil 1,9.75%,16.77%,0.64,0.79,-39.95%,-5.44%,-2.78%,-4.42%
Decil 2,11.57%,16.58%,0.74,0.93,-37.15%,-4.74%,-2.74%,-4.30%
Decil 3,8.39%,16.61%,0.57,0.71,-37.97%,-6.23%,-2.66%,-4.25%
Decil 4,12.52%,17.98%,0.75,0.96,-35.39%,-6.39%,-2.87%,-4.43%
Decil 5,11.09%,17.74%,0.68,0.85,-40.59%,-6.42%,-2.93%,-4.57%
Decil 6,8.64%,17.77%,0.56,0.70,-38.38%,-8.72%,-2.86%,-4.48%
Decil 7,9.95%,19.47%,0.59,0.73,-39.85%,-9.31%,-3.25%,-5.07%
Decil 8,10.16%,16.70%,0.66,0.89,-31.09%,-9.17%,-2.81%,-4.00%
